[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-crops.ipynb)

# Full Project: Crop Recommendation from Soil Data

*AIBits Academy · Machine Learning End To End · Full Project*

A perfectly clean, perfectly balanced 22-crop classification problem — and a genuine BaggingClassifier ensemble-of-ensembles pushing an already-strong Random Forest to 99.45% test accuracy.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['Crop_recommendation.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> Agricultural advisory services and soil-testing labs want to turn a raw soil/climate test — nitrogen, phosphorus, potassium levels, temperature, humidity, pH, rainfall — into a single actionable recommendation: *which crop should be planted on this plot*. Farmers can't act on seven raw numbers; they can act on "plant rice." A reliable classifier converts an agronomy test into a decision-support tool.

> **Dataset**
>
> **2,200 records, 22 crops, exactly 100 samples per class, zero missing values.** Features: `N`, `P`, `K` (soil nitrogen/phosphorus/potassium, kg/ha), `temperature` (°C), `humidity` (%), `ph`, `rainfall` (mm). The source file carries no location field, so this project is presented as generic agronomic data rather than tied to a specific region.

## Step 1 — Confirm the Data Is Genuinely Clean

In [ ]:
import pandas as pd
df = pd.read_csv('Crop_recommendation.csv')
print(df.shape, df.isna().sum().sum(), "missing values")
print(df['label'].value_counts().head(3))

An IQR-based outlier check run separately *within each crop class* (not across the whole dataset — important, since a K value of 200 is a genuine outlier for rice but entirely normal for grapes) confirms zero outliers across every feature/crop combination. This is a rare "friendly" dataset compared to most of the other Full Projects in this course — worth noticing precisely because it's the exception, not the rule.

## Step 2 — Pipeline + GridSearchCV Across Four Model Families

In [ ]:
from sklearn import preprocessing, model_selection
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

x = df.drop(['label'], axis=1)
y = preprocessing.LabelEncoder().fit_transform(df['label'])
x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y)   # 1,650 train / 550 test

candidates = {
    'decision_tree': (DecisionTreeClassifier(), {'decisiontreeclassifier__splitter':['best','random']}),
    'svm':           (SVC(gamma='auto', probability=True), {'svc__C':[1,10,100,1000], 'svc__kernel':['rbf','linear']}),
    'random_forest': (RandomForestClassifier(), {'randomforestclassifier__n_estimators':[1,5,10]}),
    'knn':           (KNeighborsClassifier(), {'kneighborsclassifier__n_neighbors':[5,10,20,25]}),
}
best_estimators = {}
for name, (model, params) in candidates.items():
    pipe = make_pipeline(preprocessing.StandardScaler(), model)
    grid = model_selection.GridSearchCV(pipe, params, cv=5).fit(x_train, y_train)
    print(f"{name:14s} best CV score: {grid.best_score_:.4f}")
    best_estimators[name] = grid.best_estimator_

> **Where StandardScaler Matters, and Where It Doesn't**
>
> Every pipeline here scales features before fitting — essential for SVM and KNN, which are distance/margin-based and sensitive to raw scale differences (rainfall ranges 20–300, pH ranges 3.5–10), but has **zero effect** on Decision Tree or Random Forest, which split on thresholds within a single feature at a time and are scale-invariant. Running the same pipeline structure for all four keeps the comparison fair without needing to special-case any model.

## Step 3 — Confirm on the Held-Out Test Set

In [ ]:
# Best estimator per family, scored on the untouched test set
for name, estimator in best_estimators.items():
    print(f"{name:14s} test accuracy: {estimator.score(x_test, y_test):.4f}")

## Step 4 — Bagging an Already-Bagged Model

Random Forest is itself internally a bagging method — many trees, each trained on a bootstrap-resampled subset of rows. Wrapping the *entire tuned pipeline* in an explicit `BaggingClassifier` adds a second, outer layer of bootstrap resampling on top:

In [ ]:
from sklearn.ensemble import BaggingClassifier

base_pipe = make_pipeline(preprocessing.StandardScaler(), RandomForestClassifier(n_estimators=10))
bag_model = BaggingClassifier(estimator=base_pipe, n_estimators=100,
                               max_samples=0.8, oob_score=True, random_state=0)
bag_model.fit(x_train, y_train)
print("Bagged Random Forest test accuracy:", bag_model.score(x_test, y_test))

99.09% → 99.45%: a genuine but modest gain, because the single tuned Random Forest was already close to the achievable ceiling on this clean, well-separated dataset. The full classification report backs this up — macro-average precision of 1.00 and recall of 0.99 across all 22 crops, with only two or three crops (grapes, mango) showing any misclassification at all in the confusion matrix.

## Visualizing the Marginal Gains

CV score vs. held-out test accuracy for all four tuned models, plus the final bagged Random Forest pushing past all of them — each step buys a smaller improvement than the last.

## Key Business Takeaways

- Not every real dataset is messy — when classes are clean, balanced, and well-separated, near-perfect accuracy is both achievable and trustworthy, unlike the inflated-by-imbalance accuracy seen in other projects in this course.
- An explicit `BaggingClassifier` wrapped around an already-ensembled model (Random Forest) is a legitimate additional variance-reduction lever, distinct from simply increasing `n_estimators` within the Random Forest itself — though the marginal return shrinks once the base model is already near its ceiling.
- StandardScaler inside a pipeline is "free" to include for tree-based models (no effect) but essential for distance-based ones (SVM, KNN) — a single consistent pipeline structure avoids the risk of forgetting it where it matters.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Balanced classes

Store in `n_crops` the number of distinct crops and in `per_class` the (single) number of rows each crop has, as the lesson says the classes are perfectly balanced.

In [ ]:
n_crops = per_class = None   # TODO


In [ ]:
try:
    check("22 crops", n_crops == 22)
    check("100 samples each", per_class == 100)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
n_crops = int(df["label"].nunique())
per_class = int(df["label"].value_counts().unique()[0])

```

</details>

### Exercise 2 · Medium · Scale, then KNN

Fit `make_pipeline(StandardScaler(), KNeighborsClassifier(5))` on the lesson's training split and store its test accuracy in `acc_knn`. (The lesson's tuned k-NN scored about 0.98.)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
acc_knn = None   # TODO


In [ ]:
try:
    check("high accuracy", acc_knn > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
acc_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(5)).fit(x_train, y_train).score(x_test, y_test)

```

</details>

### Exercise 3 · Stretch · Outliers per crop, not overall

A potassium (`K`) level of 200 is extreme for rice but normal for grapes, so the IQR rule must be applied **within each crop**. Write `count_outliers(frame, col)` returning the total number of rows flagged by the 1.5×IQR rule inside their own `label` group.

In [ ]:
def count_outliers(frame, col):
    pass   # TODO


In [ ]:
try:
    def brute(frame, col):
        n = 0
        for _, g in frame.groupby("label"):
            q1, q3 = g[col].quantile(0.25), g[col].quantile(0.75)
            i = q3 - q1
            n += int(((g[col] < q1 - 1.5 * i) | (g[col] > q3 + 1.5 * i)).sum())
        return n
    check("matches a brute-force count", count_outliers(df, "K") == brute(df, "K"))
    check("a naive global rule flags a different number", count_outliers(df, "K") != int(((df["K"] < df["K"].quantile(0.25) - 1.5 * (df["K"].quantile(0.75) - df["K"].quantile(0.25))) | (df["K"] > df["K"].quantile(0.75) + 1.5 * (df["K"].quantile(0.75) - df["K"].quantile(0.25)))).sum()))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def count_outliers(frame, col):
    n = 0
    for _, g in frame.groupby("label"):
        q1, q3 = g[col].quantile(0.25), g[col].quantile(0.75)
        i = q3 - q1
        n += int(((g[col] < q1 - 1.5 * i) | (g[col] > q3 + 1.5 * i)).sum())
    return n

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Crop Recommendation from Soil Data**.*